# Clean rerun: new MLflow experiment + consistent time-series CV

This version uses a new MLflow experiment (`Panaderia Salvador - SVR TimeSeriesSplit - 20261029`). Both baseline SVR evaluation and `RandomizedSearchCV` use the same `TimeSeriesSplit(n_splits=5)` object. The selected model is registered for deployment. Date parsing is robust to JSON strings, and the HTTP request sends raw columns plus 28 historical context rows for lag generation.

**Run all cells from top to bottom.** Check database credentials and confirm the final database-upload cell before running it, because it replaces the destination table (`if_exists="replace"`).


# Datathon IV - MLOps

Here is the updated table with an additional column to differentiate between positive and negative aspects, and a note that the total points are 10.

| **Activity**                                     | **Points** | **Evaluation** |
|--------------------------------------------------|------------|----------------|
| **Pipeline and MLFlow Tasks**                    |**Total: 7**|                 |
| Create a Scikit-Learn Pipeline                   | 1.0        | ✅              |
| Launch a local MLflow server                     | 0.5        | ✅              |
| Train and Evaluate the model inside a MLFlow Run | 1.0        | ✅              |
| Log the model and the metrics using MLflow       | 1.0        | ✅              |
| Register the model in MLflow                     | 1.0        | ✅              |
| Deploy the model to a REST API                   | 1.0        | ✅              |
| Make predictions using the REST API              | 1.0        | ✅              |
| Upload the predictions to the database           | 0.5        | ✅              |
| **Code and Documentation**                       |**Total: 1**|                 |
| Code legibility                                  | 0.5        | ✅              |
| Notebook documentation (titles, subtitles, text) | 0.5        | ✅              |
| **GIT Repository**                               |**Total: 3**|                 |
| Adding a complete README.md file                 | 1.0        | ✅              |
| Adding a .gitignore and requirements.txt file    | 0.5        | ✅              |
| Using branches for development                   | 0.5        | ✅              |
| **Bonus Points**                                 | **Bonus**           |                 |
| Any other activity (e.g., GridSearch)            | *Bonus*    | ✅              |
| Trying with other products                       | *Bonus*    | ✅              |
| **Negative Evaluations**                         |**Total: -6.5**|                 |
| Pushing data to the repository                   | -1.0          | ❌              |
| Pushing passwords or sensitive information       | -3.0        | ❌              |
| Pushing files generated by MLFlow (e.g., `mlruns`) | -1.0        | ❌              |
| Missing standard Python files (e.g., `requirements.txt`, `.gitignore`) | -0.5 | ❌              |
| A bad-looking README.md file                     | -1.0          | ❌              |

### Notes:
- **Total Points: 10** (excluding bonuses and negative impacts). 
- Bonus points can enhance the score but do not contribute to exceeding the maximum of 10 points. 
- Negative practices result in deductions.

# Data request, processing and modelling pipeline

Here we will connect to the database and process the raw files to build the dataset before creating our model. This notebook represents a quick workflow of the steps observed in the previous notebooks with the idea of building a pipeline to automatize all the workd. To do that, this pipeline will: 

- Connect to the database
- Add time, weather information and "festivo" days information
- Process variables (codification, scaling) --> As we saw in the EDA analysis, there are no "impossible" selling amount values, so we will not do outliers detection
- Split data into training and test datasets
- Test several models (regression models) and parameters (data splitting proportion, hiperparameters...)
- Track modelling metrics (RMSE, etc...) in MLflow

In [ ]:
# Avoid unnecessary warnings

import warnings
warnings.filterwarnings('ignore')

## 1. Prepare the data

First step always is data loading. CSV are provided and can be replicated from the previows lesson. CSVs contains data for the PRODUCT_ID 3960.

- 👉 Load train and test datasets.
- 👉 Filter dataset for producto "3960" in BOLLERIA category.
- 👉 Set "fecha_venta" as index column.
- 👉 Sort the data by "fecha_venta" column.
- 👉 Show the first 5 rows of the train dataset.

### 1.1. Load the data


Load the sensitive information within environmental variables that will be not shown as .env file is included in .gitignore

In [ ]:
import os
from dotenv import load_dotenv

# Load variables from the .env file in the project root
load_dotenv()

USERNAME = os.environ["DB_USERNAME"]
PASSWORD = os.environ["DB_PASSWORD"]
DB_HOST = os.environ["DB_HOST"]
DB_PORT = int(os.environ["DB_PORT"])
DB_NAME = os.environ["DB_NAME"]

LOCATION = (
    float(os.environ["SHOP_LAT"]),
    float(os.environ["SHOP_LON"]),
)

MLFLOW_TRACKING_URI = os.environ["MLFLOW_TRACKING_URI"]
MLFLOW_EXPERIMENT_NAME = os.environ["MLFLOW_EXPERIMENT_NAME"]

In [ ]:
required_vars = [
    "DB_USERNAME", "DB_PASSWORD", "DB_HOST", "DB_PORT", "DB_NAME",
    "SHOP_LAT", "SHOP_LON",
]

missing = [var for var in required_vars if var not in os.environ]
if missing:
    raise EnvironmentError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Check that your .env file exists and is loaded."
    )

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

print("User:", os.getenv("DB_USERNAME"))
print("Password is set:", bool(os.getenv("DB_PASSWORD")))
print("Password length:", len(os.getenv("DB_PASSWORD", "")))
print("Host:", os.getenv("DB_HOST"))
print("Port:", os.getenv("DB_PORT"))
print("Database:", os.getenv("DB_NAME"))

Load the data

In [ ]:
import os
import pandas as pd
import sqlalchemy as sa


class DatabaseConnection:
    def __init__(
        self,
        username: str,
        password: str,
        database: str | None = None,
        host: str | None = None,
        port: int | None = None,
        dialect: str = "mysql",
        driver: str = "pymysql",
    ) -> None:
        database = database or os.environ["DB_NAME"]
        host = host or os.environ["DB_HOST"]
        port = port or int(os.environ["DB_PORT"])

        connection_string = (
            f"{dialect}+{driver}://{username}:{password}"
            f"@{host}:{port}/{database}"
        )

        self.engine = sa.create_engine(connection_string)

    def query_to_df(self, query: str) -> pd.DataFrame:
        with self.engine.connect() as conn:
            result = conn.execute(sa.text(query))
            return pd.DataFrame(result.fetchall(), columns=result.keys())

    def insert_dataframe(self, df: pd.DataFrame, table_name: str) -> None:
        df.to_sql(table_name, self.engine, if_exists="replace", index=False)

    def check_connection(self) -> bool:
        try:
            with self.engine.connect() as conn:
                conn.execute(sa.text("SELECT 1"))
            return True
        except Exception as exc:
            print(f"Connection failed: {exc}")
            return False

    def close(self) -> None:
        self.engine.dispose()

In [ ]:
query = """
    SELECT familia,
    articulo,
    fechaVenta AS fecha_venta,
    festivo,
    precio,
    cantidad,
    importe,
    orden_articulo_familia

    FROM ventas_diarias_estudio
"""

database = DatabaseConnection(USERNAME, PASSWORD)


df = database.query_to_df(query)
df["fecha_venta"] = pd.to_datetime(df["fecha_venta"])
df["articulo"] = df["articulo"].astype(int)
df.head()

### 1.2. Filter by Item 

We selected the PRODUCT_ID 3960, but we can select any other item.

In [ ]:
PRODUCT_FAMILY = "BOLLERIA"
PRODUCT_ID = 3960


item_df = df.copy()
item_df = item_df[item_df["articulo"] == PRODUCT_ID]
item_df.sort_values("fecha_venta", inplace=True)
item_df.reset_index(inplace=True, drop=True)
item_df.head()

### 1.3. Transformers of the data

#### 1.3.1 Calendar Transformer

The calendar transformer will perform the following operations:

- 👉 Extract the day
- 👉 Extract the month
- 👉 Extract the year
- 👉 Extract the day of the week
- 👉 Extract the weekend (true/false)
- 👉 Extract the holiday (true/false)

First we add the `day`, `month`, `year`, `dayofweek` and `weekend` columns.

In [ ]:
def add_date_columns(df: pd.DataFrame, date_column: str = "fecha_venta") -> pd.DataFrame:
    df = df.copy()
    df[date_column] = pd.to_datetime(df[date_column], errors="raise")
    df["day"] = df[date_column].dt.day
    df["month"] = df[date_column].dt.month
    df["year"] = df[date_column].dt.year
    df["dayofweek"] = df[date_column].dt.dayofweek + 1
    df["isWeekend"] = df["dayofweek"] >= 6
    return df

item_df_tmp = item_df.copy()
add_date_columns(item_df_tmp).head()


Then we add the `holiday` column.

In [ ]:
def add_holiday_column(df: pd.DataFrame, holiday_column_name: str = "festivo") -> pd.DataFrame:
    df["isHoliday"]  = ~df[holiday_column_name].isna()
    return df

item_df_tmp = item_df.copy()
add_holiday_column(item_df_tmp).head()

Finally we create the `CalendarTransformer` class.

In [ ]:

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

class CalendarTransformer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        self.is_fitted_ = True
        return self

    def transform(self, X):
        check_is_fitted(self, "is_fitted_")

        calendar_df = add_date_columns(X)
        calendar_df = add_holiday_column(calendar_df)

        return calendar_df[
            ["day", "month", "year", "dayofweek", "isWeekend", "isHoliday"]
        ]

    def set_output(self, *, transform=None):
        if transform not in (None, "default", "pandas"):
            raise ValueError(f"Unsupported output format: {transform}")
        self._output_transform = transform
        return self


Let's transform the data.

In [ ]:
item_df_tmp = item_df.copy()

calendar_transformer = CalendarTransformer()

calendar_transformer.fit(item_df_tmp)

calendar_transformer.transform(item_df_tmp).head()

#### 1.3.2. Weather Transformer

We use the meteostat API to get the weather data. The weather transformer will perform the following operations:

- 👉 Add the weather conditions for the dates in the dataframe
- 👉 Add a `rain` column (true/false)

First we create a function to fetch the weather dataframe.

In [ ]:
import pandas as pd
import meteostat as ms
from datetime import datetime


# Location of Panadería Salvador, Málaga (latitude, longitude)
LOCATION = (36.721477644071705, -4.363132134392174)


def get_weather_data(
    start: datetime,
    end: datetime,
    location: tuple[float, float]
) -> pd.DataFrame:

    # Define shop location
    point = ms.Point(*location)

    # Find nearby weather stations
    stations = ms.stations.nearby(point, limit=5)

    if stations is None or stations.empty:
        raise ValueError("No weather stations found near the shop.")

    # Try the nearest station first, then other nearby stations
    for station_id in stations.index:
        weather_df = ms.daily(station_id, start, end).fetch()

        if weather_df is not None and not weather_df.empty:
            print(f"Using weather station: {station_id}")
            return weather_df.reset_index()

    raise ValueError(
        "No weather observations found at the nearby stations "
        "for the selected date range."
    )


def merge_by_columns(
    df1: pd.DataFrame,
    df2: pd.DataFrame,
    col_1_name: str,
    col_2_name: str
) -> pd.DataFrame:

    # Preserve the original index
    original_index = df1.index

    left = df1.copy()
    left["_original_index"] = range(len(left))

    merged_df = left.merge(
        df2,
        left_on=col_1_name,
        right_on=col_2_name,
        how="left"
    )

    merged_df = merged_df.set_index("_original_index")
    merged_df.index = original_index.take(merged_df.index)
    merged_df.index.name = df1.index.name

    return merged_df


def add_weather_data(
    df: pd.DataFrame,
    location: tuple[float, float] = LOCATION,
    date_column_name: str = "fecha_venta"
) -> pd.DataFrame:

    df = df.copy()
    df[date_column_name] = pd.to_datetime(df[date_column_name])

    start = df[date_column_name].min().normalize()
    end = df[date_column_name].max().normalize()

    weather_df = get_weather_data(start, end, location)

    # Normalize dates so sales and weather dates match
    df[date_column_name] = df[date_column_name].dt.normalize()
    weather_df["time"] = pd.to_datetime(weather_df["time"]).dt.normalize()

    return merge_by_columns(
        df, weather_df, date_column_name, "time"
    )


# Retrieve weather data and merge with sales data
weather = add_weather_data(item_df)

display(weather.head())


In [ ]:
start = item_df["fecha_venta"].min()
end = item_df["fecha_venta"].max()

item_df_tmp = item_df.copy()
weather = add_weather_data(item_df_tmp, LOCATION)
weather.head()

Add a `isRaining` column where precipitation is greater than 0.

In [ ]:

import numpy as np
import pandas as pd

def add_rain_column(
    df: pd.DataFrame,
    precipitation_column_name: str = "prcp"
) -> pd.DataFrame:

    df = df.copy()

    rain_condition = (
        df[precipitation_column_name].gt(0).fillna(False)
    )

    df["isRaining"] = rain_condition

    return df


weather_tmp = weather.copy()
weather_tmp = add_rain_column(weather_tmp)

weather_tmp.head()


Finally we create the `WeatherTransformer` class.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

class WeatherTransformer(BaseEstimator, TransformerMixin):

    def __init__(self, location: tuple[float, float]):
        self.location = location

    def fit(self, X, y=None):
        self.is_fitted_ = True
        return self

    def transform(self, X):
        check_is_fitted(self, "is_fitted_")

        weather_df = add_weather_data(X, self.location)
        weather_df = add_rain_column(weather_df)

        weather_df = weather_df[
            ["temp", "tmin", "tmax", "isRaining"]
        ]

        return weather_df

Let's transform the data.

In [ ]:
item_df_tmp = item_df.copy()

weather_transformer = WeatherTransformer(LOCATION)

weather_transformer.fit(item_df_tmp)

weather_transformer.transform(item_df_tmp).head()

#### 1.3.3. To Supervised Transformer

In order to predict the sales, we need to transform the data into a supervised problem. The `ToSupervisedTransformer` will perform the following operations:

- 👉 Obtain the `cantidad` from previous **days** and add them to the present day (shift).
- 👉 Obtain the `cantidad` from previous **weeks** and add them to the present day (shift).

In [ ]:
def series_to_supervised(
    data: pd.Series,
    daily_lag: int = 1,
    weekly_lag: int = 1,
) -> pd.DataFrame:
    """
    Converts a time series dataset into a supervised learning dataset.
    
    Args:
        data: Sequence of observations as a numpy array or DataFrame.
        daily_lag: Number of daily lag observations to include as inputs (X).
        weekly_lag: Number of weekly lag observations to include as inputs (X).
    
    Returns:
        Pandas DataFrame suitable for supervised learning.
    """
    all_features, column_names = [], []

    def create_shifted_feature(lag: int, frequency: int, freq_label: str):
        """Create lagged features for the given lag and frequency."""
        shifted_feature = data.shift(lag * frequency)
        feature_name = f"{data.name} (t-{lag} {freq_label})"
        all_features.append(shifted_feature)
        column_names.append(feature_name)

    # Add weekly lagged features
    for lag in range(weekly_lag, 0, -1):
        create_shifted_feature(lag, 7, "weeks")

    # Add daily lagged features
    for lag in range(daily_lag, 0, -1):
        create_shifted_feature(lag, 1, "days")

    # Combine all features into a single DataFrame
    df = pd.concat(all_features, axis=1)
    df.columns = column_names
    return df

In [ ]:
series_to_supervised(item_df["cantidad"], daily_lag=6, weekly_lag=4).head()

Finally we create the `ToSupervisedTransformer` class.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin, OneToOneFeatureMixin

class ToSupervisedTransformer(BaseEstimator, TransformerMixin, OneToOneFeatureMixin):
    def __init__(self, daily_lag: int, weekly_lag: int):
        self.daily_lag = daily_lag
        self.weekly_lag = weekly_lag

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return series_to_supervised(X["cantidad"], self.daily_lag, self.weekly_lag)

Let's transform the data.

In [ ]:
item_df_tmp = item_df.copy()
to_supervised_transformer = ToSupervisedTransformer(daily_lag=6, weekly_lag=4)
to_supervised_transformer.transform(item_df).head()

## 2. Create a Pipeline for all the transformations

1. Stack all the transformers in a pipeline using FeatureUnion from sklearn (preprocessor).
2. Encode the input data using OneHotEncoder and the MinMaxScaler.
3. Impute the missing values using the SimpleImputer.
4. Add a model to the pipeline.
5. Encode the target data using the MinMaxScaler.

### 2.1. Stack all the transformers: the preprocessor

In [ ]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone

class DataFrameFeatureUnion(BaseEstimator, TransformerMixin):
    def __init__(self, transformer_list):
        self.transformer_list = transformer_list

    def fit(self, X, y=None):
        self.transformers_ = []

        for name, transformer in self.transformer_list:
            fitted_transformer = clone(transformer)
            fitted_transformer.fit(X, y)
            self.transformers_.append((name, fitted_transformer))

        return self

    def transform(self, X):
        results = []

        for name, transformer in self.transformers_:
            result = transformer.transform(X)

            if not isinstance(result, pd.DataFrame):
                raise TypeError(
                    f"{name} returned {type(result)}, not a DataFrame."
                )

            results.append(result)

        return pd.concat(results, axis=1)

In [ ]:
preprocessor = DataFrameFeatureUnion(
    transformer_list=[
        ("calendar_transformer", calendar_transformer),
        ("weather_transformer", weather_transformer),
        ("to_supervised_transformer", to_supervised_transformer),
    ]
)

### 2.2. Encode the input data

Use the `OneHotEncoder` and the `MinMaxScaler`.

In [ ]:
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer

categorical_features = [
    "day",
    "month",
    "year",
    "dayofweek",
    "isWeekend",
    "isHoliday",
    "isRaining",
]

numerical_features = [
    "temp",
    "tmin",
    "tmax",
    "cantidad (t-4 weeks)",
    "cantidad (t-3 weeks)",
    "cantidad (t-2 weeks)",
    "cantidad (t-1 weeks)",
    "cantidad (t-6 days)",
    "cantidad (t-5 days)",
    "cantidad (t-4 days)",
    "cantidad (t-3 days)",
    "cantidad (t-2 days)",
    "cantidad (t-1 days)",
]

ohe_encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

mme_scaler = MinMaxScaler(feature_range=(0, 1))

data_scaler = ColumnTransformer(
    transformers=[
        ("ohe", ohe_encoder, categorical_features),
        ("scaler", mme_scaler, numerical_features),
    ],
)


### 2.3. Manage the Missing values

We can't modify the target data, so we only impute the missing values in the input data.

In [ ]:
from sklearn.impute import SimpleImputer


mean_imputer = SimpleImputer(strategy="mean")

### 2.4. Add a Model

We will use several models.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR

model = SVR()

### 2.5. Create the pipeline

Append the preprocessor, the data_scaler, the imputer and the model.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor

input_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("data_scaler", data_scaler),
        ("imputer", mean_imputer),
        ("model", model),
    ]
)

#input_pipeline.set_output(transform="pandas")

final_pipeline = TransformedTargetRegressor(
    regressor=input_pipeline,
    transformer=mme_scaler,
)

final_pipeline

### 2.6. Encode the target data

Finally use the same encoder used for the input data to encode the target data but not to fit it.

In [ ]:
from sklearn.compose import TransformedTargetRegressor


final_pipeline = TransformedTargetRegressor(
    regressor=input_pipeline,
    transformer=mme_scaler,
)
final_pipeline

## 3. Train the Model

We set a horizont for the test set of 30 days.

### 3.1. Split into Train and Test datasets.

In [ ]:
horizont = datetime(2023,4,19)


item_df_tmp = item_df.copy()
df_train = item_df.loc[item_df_tmp["fecha_venta"]<horizont]
df_test = item_df.loc[item_df_tmp["fecha_venta"]>=horizont]


X_train = df_train
y_train = df_train["cantidad"]
X_test = df_test
y_test = df_test["cantidad"]

## 4. Start MLFlow Server

- 👉 Launch a local MLFlow server
- 👉 Connect to local MLFlow server
- 👉 Set the desired experiment
- 👉 Enable MLFlow autologing for sklearn

In [ ]:
import mlflow

# Use a fresh experiment so this rerun is separate from previous experiments.
MLFLOW_EXPERIMENT_NAME = "Panaderia Salvador - SVR TimeSeriesSplit - 20261029"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
mlflow.sklearn.autolog()

print("MLflow tracking URI:", MLFLOW_TRACKING_URI)
print("MLflow experiment:", MLFLOW_EXPERIMENT_NAME)


## 5. Train and evaluate the model

The next section is to train and evaluate the model. We will use a pipeline to preprocess the data and train the model.

- 👉 Start a run in MLFlow
- 👉 Train the model using the train dataset
- 👉 Add convenient tags for PRODUCT_ID and FAMILY_ID
- 👉 Evaluate the model
- 💡 Remember this is a regression problem
- 💡 Autolog will automatically log metrics and model

### 5. Run the model


In [ ]:
from sklearn import metrics
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

# Shared CV strategy for baseline SVR evaluation and hyperparameter search.
cv = TimeSeriesSplit(n_splits=5)

with mlflow.start_run(run_name="SVR_baseline_TimeSeriesSplit") as run:
    mlflow.set_tag("product_id", PRODUCT_ID)
    mlflow.set_tag("product_family", PRODUCT_FAMILY)
    mlflow.log_param("model_type", "SVR")
    mlflow.log_param("cv_strategy", "TimeSeriesSplit")
    mlflow.log_param("cv_splits", cv.n_splits)

    final_pipeline.fit(X_train, y_train)
    y_pred = final_pipeline.predict(X_test)

    r2_cross_val = cross_val_score(
        final_pipeline, X_train, y_train, cv=cv, scoring="r2", n_jobs=1
    )
    mlflow.log_metric("r2_cross_val", r2_cross_val.mean())
    mlflow.log_metric("r2_cross_val_std", r2_cross_val.std())

    r2_test = metrics.r2_score(y_test, y_pred)
    mlflow.log_metric("r2_test", r2_test)

    print("Baseline run ID:", run.info.run_id)
    print(f"Baseline CV R²: {r2_cross_val.mean():.4f} ± {r2_cross_val.std():.4f}")
    print(f"Baseline test R²: {r2_test:.4f}")


### 5.2. Evaluate the model: Plot Results vs Predictions

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))

ax.set_title("Predicted Sales vs Actual Sales")
ax.set_xlabel("Predicted Sales")
ax.set_ylabel("Actual Sales")

sns.scatterplot(x=y_pred, y=y_test, ax=ax, label="Predicted Sales")
sns.lineplot(x=y_test, y=y_test, color="red", ax=ax, label="Real Sales")

# Add R² values to the plot
ax.text(
    0.05, 0.95,
    f"Test R² = {r2_test:.3f}\nCV R² = {r2_cross_val.mean():.3f}",
    transform=ax.transAxes,
    fontsize=11,
    verticalalignment="top",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.set_title("Predicted Sales vs Actual Sales")
ax.set_xlabel("Date")
ax.set_ylabel("Sales")
ax.set_xticklabels(df_test["fecha_venta"], rotation=45)

dates = df_test["fecha_venta"].astype(str)
sns.lineplot(x=dates, y=y_pred, label="Predicted Sales")
sns.lineplot(x=dates, y=y_test, label="Real Sales")

### 5.4. Optimize the model

In [ ]:
from scipy.stats import loguniform
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "regressor__model__kernel": ["rbf", "linear"],
    "regressor__model__C": loguniform(1e-3, 1e4),
    "regressor__model__epsilon": loguniform(1e-3, 5),
    "regressor__model__gamma": [
        "scale", "auto", 1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1, 10
    ],
}

N_ITER_SEARCH = 40

search = RandomizedSearchCV(
    estimator=clone(final_pipeline),
    param_distributions=param_distributions,
    n_iter=N_ITER_SEARCH,
    scoring={
        "mae": "neg_mean_absolute_error",
        "r2": "r2",
        "rmse": "neg_root_mean_squared_error",
    },
    refit="mae",
    cv=cv,  # Same TimeSeriesSplit instance as the baseline SVR evaluation
    n_jobs=1,
    random_state=42,
    verbose=1,
    return_train_score=True,
)


In [ ]:
MODEL_NAME = "bakery_sales_pred_SVR_25102026"

with mlflow.start_run(run_name="SVR_RandomizedSearchCV_TimeSeriesSplit") as run:
    mlflow.set_tag("product_id", PRODUCT_ID)
    mlflow.set_tag("product_family", PRODUCT_FAMILY)

    mlflow.log_param("model_type", "SVR")
    mlflow.log_param("optimization_method", "RandomizedSearchCV")
    mlflow.log_param("cv_strategy", "TimeSeriesSplit")
    mlflow.log_param("cv_splits", cv.n_splits)
    mlflow.log_param("n_iter", N_ITER_SEARCH)
    mlflow.log_param("search_scoring", "neg_mean_absolute_error")
    mlflow.log_param("random_state", 42)

    # RandomizedSearchCV refits the best configuration on all X_train.
    search.fit(X_train, y_train)
    best_pipeline = search.best_estimator_
    y_pred = best_pipeline.predict(X_test)

    mlflow.log_params({f"best_{key}": value for key, value in search.best_params_.items()})
    best_cv_mae = -search.best_score_
    mlflow.log_metric("cv_mae_best", best_cv_mae)

    # Evaluate R² with the same time-aware splits used during the search.
    r2_cv_scores = cross_val_score(
        clone(best_pipeline), X_train, y_train, cv=cv, scoring="r2", n_jobs=1
    )
    mlflow.log_metric("r2_cross_val", r2_cv_scores.mean())
    mlflow.log_metric("r2_cross_val_std", r2_cv_scores.std())

    r2_test = metrics.r2_score(y_test, y_pred)
    mae_test = metrics.mean_absolute_error(y_test, y_pred)
    rmse_test = metrics.root_mean_squared_error(y_test, y_pred)
    mlflow.log_metric("r2_test", r2_test)
    mlflow.log_metric("mae_test", mae_test)
    mlflow.log_metric("rmse_test", rmse_test)

    # Log and register the selected, fitted pipeline for deployment.
    model_info = mlflow.sklearn.log_model(
        sk_model=best_pipeline,
        artifact_path="model",
        serialization_format="cloudpickle",
        registered_model_name=MODEL_NAME,
    )

    print("MLflow run ID:", run.info.run_id)
    print("Model URI:", model_info.model_uri)
    print("Registered model:", MODEL_NAME)
    print("Best parameters:", search.best_params_)
    print(f"Best CV MAE: {best_cv_mae:.4f}")
    print(f"CV R²: {r2_cv_scores.mean():.4f} ± {r2_cv_scores.std():.4f}")
    print(f"Test R²: {r2_test:.4f}")
    print(f"Test MAE: {mae_test:.4f}")
    print(f"Test RMSE: {rmse_test:.4f}")


### 5.5. Plot again the prediction

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))

ax.set_title("Predicted Sales vs Actual Sales")
ax.set_xlabel("Predicted Sales")
ax.set_ylabel("Actual Sales")

sns.scatterplot(x=y_pred, y=y_test, ax=ax, label="Predicted Sales")
sns.lineplot(x=y_test, y=y_test, color="red", ax=ax, label="Real Sales")

# Add R² values to the plot
ax.text(
    0.05, 0.95,
    f"Test R² = {r2_test:.3f}\nCV R² = {r2_cross_val.mean():.3f}",
    transform=ax.transAxes,
    fontsize=11,
    verticalalignment="top",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
)

plt.tight_layout()
plt.show()

## 6. Deploy the model

After running the model-search cell, check the latest registered version in the output of the model registry cell below. Stop any old model-serving process first. Keep the MLflow tracking server on port 5000 and serve the model on port 5001.

In a terminal, replace `<NEW_VERSION>` with the latest version printed by the registry cell:

```bash
export MLFLOW_TRACKING_URI=http://localhost:5000
mlflow models serve \
  -m "models:/bakery_sales_pred_SVR_25102026/<NEW_VERSION>" \
  -p 5001 \
  --env-manager local
```

The prediction endpoint is `http://127.0.0.1:5001/invocations`.


In [ ]:
import mlflow
from mlflow import MlflowClient

mlflow.set_tracking_uri("http://localhost:5000")
client = MlflowClient()

for model in client.search_registered_models():
    print(f"\nModel: {model.name}")
    for version in model.latest_versions:
        print(
            f"  Version: {version.version}, "
            f"Aliases: {version.aliases}, "
            f"Status: {version.status}"
        )

## 7. Make requests to the model

The model is now deployed and ready to receive requests. We will make a request to the model using the test set.

- 👉 prepare the test set to be sent as JSON
- 👉 make a POST request to the model
- 👉 get the predictions from the response and show them

In [ ]:
import requests
import json

URL = "http://127.0.0.1:5001/invocations"
headers = {"Content-Type": "application/json"}

raw_columns = [
    "familia", "articulo", "fecha_venta", "festivo",
    "precio", "cantidad", "importe", "orden_articulo_familia",
]

# The pipeline computes lag features from rows it receives. Add 28 historical rows
# so the first test predictions can use the previous four weeks of observations.
df_request = pd.concat(
    [df_train.tail(28)[raw_columns], df_test[raw_columns]],
    ignore_index=True,
).copy()
df_request["fecha_venta"] = pd.to_datetime(
    df_request["fecha_venta"], errors="raise"
).dt.strftime("%Y-%m-%d")

payload = {
    "dataframe_split": {
        "columns": df_request.columns.tolist(),
        "index": df_request.index.tolist(),
        "data": df_request.values.tolist(),
    }
}

response = requests.post(URL, headers=headers, json=payload, timeout=120)
response.raise_for_status()
all_predictions = response.json()["predictions"]
predictions = all_predictions[-len(df_test):]

# Keep the test-only frame for the database upload in Step 8.
df_send = df_test[raw_columns].copy()
df_send["fecha_venta"] = pd.to_datetime(
    df_send["fecha_venta"], errors="raise"
).dt.strftime("%Y-%m-%d")

assert len(predictions) == len(df_test), (
    f"Expected {len(df_test)} predictions, got {len(predictions)}"
)
print("HTTP status:", response.status_code)
print("Predictions returned for test rows:", len(predictions))
print(predictions[:5])


## 8. Push Results to Database

We push the results to the database so we can visualize them using other tools like Tableau, PowerBI, etc.

Prepare the dataframe to upload to the database

In [ ]:
dates = df_send["fecha_venta"].tolist()

# Create a dataframe with the data to store
df_article_prediction = pd.DataFrame({
    "fecha": dates,
    "cantidad": predictions,
    "articulo": [PRODUCT_ID] * len(predictions),  # repeat the article for each date
    "familia": [PRODUCT_FAMILY] * len(predictions),  # repeat the family for each date
})

df_article_prediction.head(5)

Push the dataframe to the database

In [ ]:
# Database credentials
DATABASE = DB_NAME
NAME = USERNAME

table_name = f"Materials_Prediction_Group_{NAME}"

# Connect to the database
db = DatabaseConnection(USERNAME, PASSWORD, database=DB_NAME)
db.insert_dataframe(df_article_prediction, table_name)
